In [1]:
import ee
import pandas as pd
import numpy as np

C:\Users\Projit\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.5) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# 1. Authenticate and initialize GEE
# ==========================================
ee.Authenticate()
ee.Initialize(project="sundarbans-climate-analysis")

In [3]:
# 2. Study area: approximate coastal region
# west, south, east, north
# ==========================================
roi = ee.Geometry.Rectangle(
    [88.5, 20.5, 90.5, 22.0]
)

In [4]:
# 3. Load historical Copernicus salinity data
# ==========================================
collection = (
    ee.ImageCollection(
        "COPERNICUS/MARINE/GLOBAL_SEA_SURFACE/REP_DAILY"
    )
    .filterDate("2000-01-01", "2025-01-01")
    .select("sos")
)

print(
    "Available daily images:",
    collection.size().getInfo()
)

Available daily images: 9116


In [5]:
# 4. Create monthly date sequence
# ==========================================
start_date = ee.Date("2000-01-01")
month_count = 300  # Jan 2000 to Dec 2024

months = ee.List.sequence(0, month_count - 1)



In [6]:
# ==========================================
# 5. Calculate monthly mean salinity
# ==========================================
def calculate_monthly_salinity(index):
    month_start = start_date.advance(index, "month")
    month_end = month_start.advance(1, "month")

    monthly_images = collection.filterDate(
        month_start, month_end
    )

    image_count = monthly_images.size()

    monthly_mean = monthly_images.mean().rename("sos")

    stats = monthly_mean.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=14000,
        maxPixels=1e8,
        bestEffort=True
    )

    return ee.Feature(None, {
        "Year": month_start.get("year"),
        "Month": month_start.get("month"),
        "Date": month_start.format("YYYY-MM"),
        "Mean_Salinity_PSU": stats.get("sos"),
        "Available_Image_Count": image_count
    })

monthly_fc = ee.FeatureCollection(
    months.map(calculate_monthly_salinity)
)

# Remove months with no usable observations
monthly_fc = monthly_fc.filter(
    ee.Filter.gt("Available_Image_Count", 0)
).filter(
    ee.Filter.notNull(["Mean_Salinity_PSU"])
)


In [7]:
# 6. Transfer the monthly table to Python
# ==========================================
result = monthly_fc.getInfo()

rows = [
    feature["properties"]
    for feature in result["features"]
]

df = pd.DataFrame(rows)

df = df.sort_values(
    ["Year", "Month"]
).reset_index(drop=True)


In [8]:
# 7. Export to Excel
# ==========================================
output_file = "Sundarbans_Monthly_Salinity_2000_2024.xlsx"

with pd.ExcelWriter(
    output_file, engine="openpyxl"
) as writer:
    df.to_excel(
        writer,
        sheet_name="Monthly_Salinity",
        index=False
    )

print("\nExcel file created successfully!")
print("File:", output_file)
print("Monthly records:", len(df))
print(df.head(12).to_string(index=False))



Excel file created successfully!
File: Sundarbans_Monthly_Salinity_2000_2024.xlsx
Monthly records: 300
 Available_Image_Count    Date  Mean_Salinity_PSU  Month  Year
                    31 2000-01          29.923017      1  2000
                    29 2000-02          30.446180      2  2000
                    31 2000-03          31.185842      3  2000
                    30 2000-04          31.746893      4  2000
                    31 2000-05          31.972457      5  2000
                    30 2000-06          31.669591      6  2000
                    31 2000-07          30.855372      7  2000
                    31 2000-08          29.630122      8  2000
                    30 2000-09          28.654630      9  2000
                    31 2000-10          28.254020     10  2000
                    30 2000-11          28.670657     11  2000
                    31 2000-12          29.283752     12  2000


In [9]:
## EDA for salainity

In [10]:
import os

In [11]:
input_file = "Sundarbans_Monthly_Salinity_2000_2024.xlsx"

In [12]:
os.makedirs("EDA", exist_ok=True)

In [13]:
output_file = os.path.join(
    "EDA",
    "Salinity_EDA_2_1_Descriptive_Statistics.xlsx"
)

In [14]:
df = pd.read_excel(input_file)

In [15]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Mean_Salinity_PSU"] = pd.to_numeric(
    df["Mean_Salinity_PSU"], errors="coerce"
)

In [16]:
salinity = df["Mean_Salinity_PSU"].dropna()

In [17]:
## DATASET OVERVIEW

In [18]:
overview = pd.DataFrame({
    "Metric": [
        "Total Observations",
        "Total Variables",
        "Start Date",
        "End Date",
        "Missing Salinity Values",
        "Duplicate Records",
        "Minimum Salinity (PSU)",
        "Maximum Salinity (PSU)",
        "Mean Salinity (PSU)"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df["Date"].min(),
        df["Date"].max(),
        df["Mean_Salinity_PSU"].isna().sum(),
        df.duplicated().sum(),
        salinity.min(),
        salinity.max(),
        salinity.mean()
    ]
})

In [19]:
## DESCRIPTIVE STATISTICS

In [20]:
descriptive_stats = salinity.describe().to_frame(
    name="Salinity_PSU"
)

descriptive_stats.loc["Range"] = (
    salinity.max() - salinity.min()
)

descriptive_stats.loc["IQR"] = (
    salinity.quantile(0.75) -
    salinity.quantile(0.25)
)

descriptive_stats.loc["Variance"] = salinity.var()
descriptive_stats.loc["Standard Deviation"] = salinity.std()


In [21]:
## DISTRIBUTION SHAPE

In [22]:
distribution = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Skewness",
        "Kurtosis"
    ],
    "Value": [
        salinity.mean(),
        salinity.median(),
        salinity.skew(),
        salinity.kurt()
    ]
})


In [23]:
## IQR-BASED OUTLIER DETECTION

In [24]:
Q1 = salinity.quantile(0.25)
Q3 = salinity.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

df["Outlier_Status"] = np.where(
    (df["Mean_Salinity_PSU"] < lower_bound) |
    (df["Mean_Salinity_PSU"] > upper_bound),
    "Potential Outlier",
    "Normal"
)

outliers = df[
    df["Outlier_Status"] == "Potential Outlier"
].copy()

outlier_summary = pd.DataFrame({
    "Metric": [
        "Q1",
        "Q3",
        "IQR",
        "Lower Bound",
        "Upper Bound",
        "Total Potential Outliers"
    ],
    "Value": [
        Q1,
        Q3,
        IQR,
        lower_bound,
        upper_bound,
        len(outliers)
    ]
})

In [25]:
## MONTHLY SUMMARY

In [27]:
monthly_summary = (
    df.groupby(["Month"])["Mean_Salinity_PSU"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Standard_Deviation="std",
        Minimum="min",
        Maximum="max"
    )
    .reset_index()
    .sort_values("Month")
)


In [28]:
## ANNUAL SUMMARY

In [29]:
annual_summary = (
    df.groupby("Year")["Mean_Salinity_PSU"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Standard_Deviation="std",
        Minimum="min",
        Maximum="max"
    )
    .reset_index()
)


In [30]:
## IMAGE COUNT SUMMARY

In [31]:
image_count_summary = df["Available_Image_Count"].describe().to_frame(
    name="Available_Image_Count"
)

In [32]:
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    overview.to_excel(
        writer, sheet_name="Dataset_Overview", index=False
    )

    descriptive_stats.to_excel(
        writer, sheet_name="Descriptive_Statistics"
    )

    distribution.to_excel(
        writer, sheet_name="Distribution_Shape", index=False
    )

    outlier_summary.to_excel(
        writer, sheet_name="Outlier_Summary", index=False
    )

    outliers.to_excel(
        writer, sheet_name="Potential_Outliers", index=False
    )

    monthly_summary.to_excel(
        writer, sheet_name="Monthly_Summary", index=False
    )

    annual_summary.to_excel(
        writer, sheet_name="Annual_Summary", index=False
    )

    image_count_summary.to_excel(
        writer, sheet_name="Image_Count_Summary"
    )

In [33]:
print("Salinity EDA 2.1 completed successfully!")
print("Report saved at:", os.path.abspath(output_file))

Salinity EDA 2.1 completed successfully!
Report saved at: C:\projects_data\sundarbans_climate_analysis_gee\EDA\Salinity_EDA_2_1_Descriptive_Statistics.xlsx
